In [1]:
import sys
sys.path.append("../src")

from OmiScraper import OmiScraper
from RealListing import RealListing
from AreasDictionaries import university_areas
from AreasDictionaries import area_university_list
from Functions import column_check, numerical_check, categorical_check

import pandas as pd
import numpy as np
from io import StringIO

## 1. Data import and initial inspection of the dataset
This first step consists of importing the dataset and performing an initial inspection to understand its structure, data types, and overall completeness.

The inspection is carried out using **.head()** and **.info()**, which allow to quickly check:
- the number of columns and records  
- the data types of each variable  
- the presence of missing values  

This step helps identify potential inconsistencies early and gives direction for the data cleaning process.

The dataset is composed of **18 columns and 43 records**, as confirmed by the inspection.


In [4]:
dt = pd.read_csv("../Datasets/Real_Listing_dt_Fiumicino.csv")

In [6]:
dt.head(5)

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,ID_number,Stree
0,Isola Sacra (FCO) - Via Trincea delle Frasche,Passo Buole,Iso_Buo_1,296000,130,4,3,B,40.0,3,3.0,O,N,2,available,2019.0,NaN,NaN
1,Isola Sacra (FCO) - Via Trincea delle Frasche,Trince delle Frasche,Iso_Fra_2,197000,60,2,1,T,0.0,1,2.0,O,N,1,available,2025.0,NaN,NaN
2,Isola Sacra (FCO) - Via Trincea delle Frasche,Trince delle Frasche,Iso_Fra_3,285000,125,4,3,B,0.0,1,2.0,B,N,1,available,2026.0,NaN,NaN
3,Isola Sacra (FCO) - Via Trincea delle Frasche,Trince delle Frasche,Iso_Fra_4,235000,98,4,3,G,0.0,1,2.0,B,N,2,Auction,2026.0,NaN,NaN
4,Isola Sacra (FCO) - Via Trincea delle Frasche,Bruno Carloni,Iso_Car_5,205000,68,3,1,T,NaN,2,2.0,B,Y,2,available,2026.0,NaN,NaN


In [8]:
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Area               43 non-null     object 
 1   Street             43 non-null     object 
 2   ID_Number          43 non-null     object 
 3   Price              43 non-null     int64  
 4   Surface            43 non-null     int64  
 5   N. of Rooms        43 non-null     int64  
 6   N. of Bathrooms    43 non-null     int64  
 7   External Area      43 non-null     object 
 8   External Surface   23 non-null     float64
 9   Floor              42 non-null     object 
 10  Levels             42 non-null     float64
 11  Parking Space      40 non-null     object 
 12  Furnished          43 non-null     object 
 13  Conditions         43 non-null     int64  
 14  Property           43 non-null     object 
 15  Construction Year  38 non-null     float64
 16  ID_number          1 non-nul

## 2. Consistency checks & data validation

### 2.1 Column deletion and duplicated check
From the initial inspection, it appeared clear that two columns (**ID_number** and **Stree**) contained typos and were incorrectly introduced in the dataset. As they do not provide meaningful information for the analysis, they were removed.

A duplicate check was then performed using **.duplicated().sum()**, which returned **0**, confirming that there are no fully duplicated records in the dataset.

This ensures that each observation represents a unique listing and that no further action is required at this stage regarding duplicate entries.

In [14]:
dt.drop(["ID_number", "Stree"], axis = 1, inplace = True)

In [16]:
dt.duplicated().sum()

0

### 2.2 Value check & missing values
Missing values and general value consistency were checked across all columns using a loop that returns:
- **unique values** 
- **number of NaN**
- **number of zeros**  
- **number of total and unique values**  

This allowed for a quick overview of the structure and potential inconsistencies in each feature.

In addition, a simple manual inspection snippet was implemented to perform the same checks on a single feature, allowing for a more direct and focused inspection when needed.

The output of this check highlights several key patterns in the dataset.  
Most core numerical features (e.g. **Price**, **Surface**, **Rooms**) are complete and do not contain missing or zero values, confirming their reliability for analysis.

However, inconsistencies are present in other features:
- **External Surface** shows a high number of missing values and some zero entries  
- **Floor** contains mixed formats (numeric values and text such as "whole building")  
- **Property** includes inconsistent labeling (e.g. capitalization and typos)  
- **Construction Year** and **Parking Space** contain missing values  

These findings guided the subsequent cleaning steps, distinguishing between simple corrections and cases requiring deeper validation.

***NOTE***: During this step, some inconsistencies were identified and corrected by directly cross-checking the values against the original listings.  
More complex inconsistencies, particularly those involving cross-column dependencies and logical relationships, were not addressed here and were instead handled in the final validation section, as they required deeper inspection.

In [19]:
column_list = dt.columns.tolist()

column_check(dt, column_list)

,Column,NaN,Zeros,Unique Values,Total Values
0,Area,0,0,1,43
1,Street,0,0,20,43
2,ID_Number,0,0,43,43
3,Price,0,0,29,43
4,Surface,0,0,33,43
5,N. of Rooms,0,0,7,43
6,N. of Bathrooms,0,0,4,43
7,External Area,0,0,9,43
8,External Surface,20,7,15,43
9,Floor,1,0,6,43


In [21]:
col_to_check = input("Which column do you want to examinate?")
while True:
    if col_to_check in column_list:
        print("-" * 40 + f"  \033[91mColumn = **{col_to_check}**\033[0m  " + "-" * 40)
        print()
        print(f"\033[92mThe list of unique values is:\033[0m\n{dt[col_to_check].unique().tolist()}")
        print()
        print(f"\033[92mThe number of NaN in is:\033[0m\n{dt[col_to_check].isna().sum()}")
        print()
        print(f"\033[92mThe number of unique values in is:\033[0m\n{dt[col_to_check].nunique()}")
        print()
        print(f"\033[92mThe number of repeated values is:\033[0m\n{dt[col_to_check].duplicated().sum()}")
        print()
        print(f"\033[92mThe number of total values is:\033[0m\n{dt[col_to_check].count() + dt[col_to_check].isna().sum()}")
        break
    else:
        col_to_check = input("Wrong column, please select an available column:")
        continue

Which column do you want to examinate? Floor


----------------------------------------  Column = **Floor**  ----------------------------------------

The list of unique values is:
['3', '1', '2', 'whole building', '0', 'whole Building', nan]

The number of NaN in is:
1

The number of unique values in is:
6

The number of repeated values is:
36

The number of total values is:
43


---

In [24]:
dt[dt["External Surface"].isna()]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year
4,Isola Sacra (FCO) - Via Trincea delle Frasche,Bruno Carloni,Iso_Car_5,205000,68,3,1,T,NaN,2,2.0,B,Y,2,available,2026.0
5,Isola Sacra (FCO) - Via Trincea delle Frasche,Athos Maestri,Iso_Mae_6,235000,126,3,1,B + T,NaN,1,1.0,O,Y,3,available,1983.0
7,Isola Sacra (FCO) - Via Trincea delle Frasche,A. Zezon,Iso_Zez_8,390000,105,4,2,T + B + G,NaN,1,2.0,O,N,2,new consruction,2026.0
14,Isola Sacra (FCO) - Via Trincea delle Frasche,Passo Buole,Iso_Buo_15,123000,43,3,2,T,NaN,2,2.0,O,N,2,auction,1980.0
15,Isola Sacra (FCO) - Via Trincea delle Frasche,Passo Buole,Iso_Buo_16,290000,142,3,2,T,NaN,2,2.0,O,N,2,available,1984.0
16,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_17,259000,136,3,1,B,NaN,1,3.0,N,N,1,available,1990.0
18,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_19,259000,123,3,2,B,NaN,2,2.0,N,N,2,available,1965.0
19,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_20,285000,120,4,2,G,NaN,1,1.0,B,N,2,available,1984.0
20,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_21,285000,120,6,1,G,NaN,2,2.0,B,N,2,available,1984.0
21,Isola Sacra (FCO) - Via Trincea delle Frasche,Via Debeli,Iso_Deb_22,259000,70,4,2,G,NaN,1,1.0,O,N,2,available,1980.0


---

In [27]:
dt[dt["Floor"].isna()]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year
26,Isola Sacra (FCO) - Via Trincea delle Frasche,Antonio Cambriglia,Iso_Cam_27,235000,110,5,1,B + T,NaN,NaN,NaN,NaN,N,1,new construction,2026.0


---

In [30]:
dt[dt["Levels"].isna()]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year
26,Isola Sacra (FCO) - Via Trincea delle Frasche,Antonio Cambriglia,Iso_Cam_27,235000,110,5,1,B + T,NaN,NaN,NaN,NaN,N,1,new construction,2026.0


---

In [33]:
dt[dt["Parking Space"].isna()]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year
24,Isola Sacra (FCO) - Via Trincea delle Frasche,Passo Buole,Iso_Buo_25,123000,77,4,1,T + B,NaN,2,3.0,NaN,P,2,auction,NaN
25,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_26,235000,98,5,1,B,NaN,0,3.0,NaN,N,2,auction,NaN
26,Isola Sacra (FCO) - Via Trincea delle Frasche,Antonio Cambriglia,Iso_Cam_27,235000,110,5,1,B + T,NaN,NaN,NaN,NaN,N,1,new construction,2026.0


In [35]:
dt.loc[dt["ID_Number"] == "Iso_Buo_25", "Parking Space"] = "N"

---

In [38]:
dt[dt["Construction Year"].isna()]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year
17,Isola Sacra (FCO) - Via Trincea delle Frasche,Alberto Brondi,Iso_Bro_18,275000,200,8,2,B + T + G,450.0,whole building,2.0,N,N,2,missing documentation,NaN
23,Isola Sacra (FCO) - Via Trincea delle Frasche,Passo Buole,Iso_Buo_24,239000,99,4,2,N,NaN,1,2.0,B,P,2,available,NaN
24,Isola Sacra (FCO) - Via Trincea delle Frasche,Passo Buole,Iso_Buo_25,123000,77,4,1,T + B,NaN,2,3.0,N,P,2,auction,NaN
25,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_26,235000,98,5,1,B,NaN,0,3.0,NaN,N,2,auction,NaN
27,Isola Sacra (FCO) - Via Trincea delle Frasche,Ferdinando Guerci,Iso_Gue_28,195000,71,2,1,B,NaN,1,2.0,N,N,2,available,NaN


In [40]:
dt.loc[dt["ID_Number"] == "Iso_Buo_25", "Construction Year"] = 1980

---

### 2.3 Numerical & data type check
Numerical features were validated to ensure correct data types and consistency of values.

The **Floor** column was initially stored as an object type due to the presence of non-numeric values (e.g. "whole building"). To allow proper numerical comparison, it was converted using pd.to_numeric() with coercion, transforming non-numeric values into NaN.

To preserve the original information, a duplicate column (**Floor1**) was created, where missing values were replaced with the original categorical value ("whole building"). This allowed both numerical validation and retention of the original data.

A structured check was then performed across all numerical features using a helper function that returns:
- **unique values**
- **minimum and maximum values**
- **data type**

This provided a quick validation of numerical ranges and helped identify potential inconsistencies.

As Before, a simple manual inspection snippet was implemented to perform the same validation on individual features, allowing for a more direct and focused inspection when needed.

Additional checks were performed to verify that numerical values stored as floats were effectively integers. Once confirmed (i.e. no meaningful decimal components), selected numerical features (**External Surface**, **Floor**, **Levels**, and **Construction Year**) were converted to integer format using the nullable **Int64** type.

This conversion ensured integer representation while preserving missing values (NaN), maintaining consistency across the dataset without loss of information.

In [44]:
dt["Floor"] = pd.to_numeric(dt["Floor"], errors = "coerce")

In [46]:
dt["Floor1"] = dt["Floor"].fillna("whole building")
dt["Floor1"].unique()

array([3.0, 1.0, 2.0, 'whole building', 0.0], dtype=object)

In [48]:
dt[(dt["Floor1"] == "whole building") | (dt["Floor1"] == "whole Building")]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,Floor1
8,Isola Sacra (FCO) - Via Trincea delle Frasche,Gaetano Carolei,Iso_Car_9,450000,170,6,2,T + B + G,300.0,NaN,2.0,B,N,1,new consruction,2026.0,whole building
9,Isola Sacra (FCO) - Via Trincea delle Frasche,Gherardo Vaiarini,Iso_Vai_10,459000,400,6,4,T + B + G,300.0,NaN,3.0,B + C,N,2,available,1980.0,whole building
12,Isola Sacra (FCO) - Via Trincea delle Frasche,Teresio Martinoli,Iso_Mar_13,249000,75,2,2,B + G,25.0,NaN,2.0,O,N,2,available,1990.0,whole building
17,Isola Sacra (FCO) - Via Trincea delle Frasche,Alberto Brondi,Iso_Bro_18,275000,200,8,2,B + T + G,450.0,NaN,2.0,N,N,2,missing documentation,NaN,whole building
22,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_23,594000,240,10,2,T + B,NaN,NaN,2.0,B,N,2,available,1984.0,whole building
26,Isola Sacra (FCO) - Via Trincea delle Frasche,Antonio Cambriglia,Iso_Cam_27,235000,110,5,1,B + T,NaN,NaN,NaN,NaN,N,1,new construction,2026.0,whole building
28,Isola Sacra (FCO) - Via Trincea delle Frasche,Foscolo Montini,Iso_Mon_29,335000,100,4,2,B + G,150.0,NaN,2.0,O,N,1,new construction,2026.0,whole building
30,Isola Sacra (FCO) - Via Trincea delle Frasche,Stellato Spalletti,Iso_Spa_31,365000,115,4,2,B + G,NaN,NaN,2.0,O,N,1,new construction,2026.0,whole building
31,Isola Sacra (FCO) - Via Trincea delle Frasche,Stellato Spalletti,Iso_Spa_32,390000,117,5,2,B + G,NaN,NaN,2.0,O,N,1,new construction,2026.0,whole building
33,Isola Sacra (FCO) - Via Trincea delle Frasche,Col Fenilon,Iso_Fen_34,349000,145,5,2,B + G,450.0,NaN,2.0,B,N,2,available,1990.0,whole building


In [50]:
numerical_list = [
    "Price", "Surface", "N. of Rooms", "N. of Bathrooms", "External Surface", "Floor", "Levels", "Conditions", "Construction Year"
]

numerical_check(dt, numerical_list)

,Column,Min Value,Max Value,Data Type
0,Price,123000.0,594000.0,int64
1,Surface,42.0,400.0,int64
2,N. of Rooms,2.0,10.0,int64
3,N. of Bathrooms,1.0,4.0,int64
4,External Surface,0.0,450.0,float64
5,Floor,0.0,3.0,float64
6,Levels,1.0,4.0,float64
7,Conditions,1.0,3.0,int64
8,Construction Year,1965.0,2026.0,float64


In [52]:
for num_col1 in numerical_list:
    print("-" * 40 + f"\033[91m{num_col1} info:\033[0m" + "-" * 40)

    print(f"\033[92mThe unique values for {num_col1} is:\033[0m\n {dt[num_col1].unique().tolist()}")
    print()

----------------------------------------Price info:----------------------------------------
The unique values for Price is:
 [296000, 197000, 285000, 235000, 205000, 390000, 450000, 459000, 139000, 249000, 123000, 290000, 259000, 275000, 594000, 239000, 195000, 335000, 209000, 365000, 349000, 279000, 339000, 220000, 174000, 175000, 225000, 200000, 250000]

----------------------------------------Surface info:----------------------------------------
The unique values for Surface is:
 [130, 60, 125, 98, 68, 126, 120, 105, 170, 400, 48, 75, 100, 43, 142, 136, 200, 123, 70, 240, 99, 77, 110, 71, 115, 117, 84, 145, 128, 55, 42, 94, 90]

----------------------------------------N. of Rooms info:----------------------------------------
The unique values for N. of Rooms is:
 [4, 2, 3, 5, 6, 8, 10]

----------------------------------------N. of Bathrooms info:----------------------------------------
The unique values for N. of Bathrooms is:
 [3, 1, 2, 4]

----------------------------------------

In [53]:
num_col1 = input("Which numerical column do you want to examinate? ")

while True: 
    if num_col1 in numerical_list:
        print("-" * 40 + f"\033[91m{num_col1} info:\033[0m" + "-" * 40)
        
        print(f"\033[92mThe unique values for {num_col1} is:\033[0m\n {dt[num_col1].unique().tolist()}")
        print()
        
        print(f"\033[92mThe range for {num_col1} is:\033[0m\nmin = {dt[num_col1].min()}\nmax = {dt[num_col1].max()}")
        print()
        
        print(f"\033[92mThe data type is:\033[0m\n{dt[num_col1].dtype}")
        break

    else:
        num_col1 = input("Please enter a valid numerical feature")

Which numerical column do you want to examinate?  Floor


----------------------------------------Floor info:----------------------------------------
The unique values for Floor is:
 [3.0, 1.0, 2.0, nan, 0.0]

The range for Floor is:
min = 0.0
max = 3.0

The data type is:
float64


In [56]:
((dt["Floor"] % 1 !=0) & (~dt["Floor"].isna())).sum()

0

In [58]:
((dt["Levels"] % 1 !=0) & (~dt["Levels"].isna())).sum()

0

In [60]:
((dt["Construction Year"] % 1 !=0) & (~dt["Construction Year"].isna())).sum()

0

In [62]:
to_int_list = ["External Surface", "Floor", "Levels", "Construction Year"]

In [64]:
for col11 in to_int_list:
    dt[col11] = dt[col11].astype("Int64")
    print("-" * 80)
    print(f"\033[92m{col11} has been converted from Float to:\033[0m {dt[col11].dtype}")

--------------------------------------------------------------------------------
External Surface has been converted from Float to: Int64
--------------------------------------------------------------------------------
Floor has been converted from Float to: Int64
--------------------------------------------------------------------------------
Levels has been converted from Float to: Int64
--------------------------------------------------------------------------------
Construction Year has been converted from Float to: Int64


In [66]:
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Area               43 non-null     object
 1   Street             43 non-null     object
 2   ID_Number          43 non-null     object
 3   Price              43 non-null     int64 
 4   Surface            43 non-null     int64 
 5   N. of Rooms        43 non-null     int64 
 6   N. of Bathrooms    43 non-null     int64 
 7   External Area      43 non-null     object
 8   External Surface   23 non-null     Int64 
 9   Floor              32 non-null     Int64 
 10  Levels             42 non-null     Int64 
 11  Parking Space      41 non-null     object
 12  Furnished          43 non-null     object
 13  Conditions         43 non-null     int64 
 14  Property           43 non-null     object
 15  Construction Year  39 non-null     Int64 
 16  Floor1             43 non-null     object
dtyp

---

### 2.4 Categorical & data type check
Categorical features were validated to ensure consistency in formatting, labeling, and structure.

A preliminary check was performed using a pre-built function to identify common issues such as leading/trailing spaces, multiple internal spaces, and capitalization inconsistencies across all categorical variables.

In addition, an interactive check was used to inspect individual features more directly when needed, allowing for a more focused validation.

To further assess consistency, the unique values of each categorical feature were examined. This step was particularly useful in identifying incoherent labels, typos, and inconsistent formatting within the same category.

Through this process, several inconsistencies were detected and corrected:
- A typo in the **Street** column ("Trince delle Frasche") was standardized to "Trincea delle Frasche"
- The ordering of combined categories in **External Area** (e.g. "T + B" → "B + T") was normalized
- Inconsistent capitalization in **Property** (e.g. "Auction" → "auction") was corrected

These corrections were based on the assumption that categorical values should be internally consistent and follow a standardized format to avoid fragmentation during analysis.

The combination of structured checks and manual inspection ensured that categorical features were clean, coherent, and ready for further analysis.

In [70]:
categorical_list = ["Area", "Street", "ID_Number", "External Area", "Parking Space", "Furnished", "Property"]

categorical_check(dt, categorical_list)

,Column,Leading Spaces,Trailing Spaces,Multiple Internal Spaces,First Capital Letter
0,Area,0,0,0,43
1,Street,0,0,0,43
2,ID_Number,0,0,0,43
3,External Area,0,0,0,43
4,Parking Space,0,0,0,41
5,Furnished,0,0,0,43
6,Property,0,0,0,1


In [72]:
cat_col2 = input("Which categorical column do you want to examinate? ")

while True:

    if  cat_col2 in categorical_list:
        print("-" * 40 + f"\033[91m {cat_col2} - Space Consistency Check \033[0m" + "-" * 40)
        
        print(f"\033[92mNumber of values with leading spaces:\033[0m\n{dt[cat_col2].str.startswith(' ').sum()}")
        print()
        
        print(f"\033[92mNumber of values with trailing spaces:\033[0m\n{dt[cat_col2].str.endswith(' ').sum()}")
        print()
        
        print(f"\033[92mNumber of values with multiple internal spaces:\033[0m\n{dt[cat_col2].str.contains(r'\s{2,}').sum()}")
        print()
        
        print(f"\033[92mNumber of values with first capital letter:\033[0m\n{dt[cat_col2].str[0].str.isupper().sum()}")
        break

    else:
        cat_col2 = input("Please enter a valid feature")
        continue


Which categorical column do you want to examinate?  Parking Space


---------------------------------------- Parking Space - Space Consistency Check ----------------------------------------
Number of values with leading spaces:
0

Number of values with trailing spaces:
0

Number of values with multiple internal spaces:
0

Number of values with first capital letter:
41


---

In [74]:
for cat_col2 in categorical_list:
    print("-" * 40 + f"\033[91m{cat_col2} - unique\033[0m" + "-" * 40)
    print(f"\033[92m{cat_col2} unique values are:\033[0m\n{dt[cat_col2].unique().tolist()}")
    print()

----------------------------------------Area - unique----------------------------------------
Area unique values are:
['Isola Sacra (FCO) - Via Trincea delle Frasche']

----------------------------------------Street - unique----------------------------------------
Street unique values are:
['Passo Buole', 'Trince delle Frasche', 'Bruno Carloni', 'Athos Maestri', 'Angelo Siffredi', 'A. Zezon', 'Gaetano Carolei', 'Gherardo Vaiarini', 'Teresio Martinoli', 'Trincea delle Frasche', 'Alberto Brondi', 'Via Debeli', 'Antonio Cambriglia', 'Ferdinando Guerci', 'Foscolo Montini', 'Stellato Spalletti', 'Sei Busi', 'Col Fenilon', 'Vittorio Marandola', 'Via Condino']

----------------------------------------ID_Number - unique----------------------------------------
ID_Number unique values are:
['Iso_Buo_1', 'Iso_Fra_2', 'Iso_Fra_3', 'Iso_Fra_4', 'Iso_Car_5', 'Iso_Mae_6', 'Iso_Sif_7', 'Iso_Zez_8', 'Iso_Car_9', 'Iso_Vai_10', 'Iso_Zez_11', 'Iso_Zez_12', 'Iso_Mar_13', 'Iso_Zez_14', 'Iso_Buo_15', 'Iso_Bu

---

In [77]:
dt[dt["Parking Space"].isna()]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,Floor1
25,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_26,235000,98,5,1,B,<NA>,0,3,NaN,N,2,auction,<NA>,0.0
26,Isola Sacra (FCO) - Via Trincea delle Frasche,Antonio Cambriglia,Iso_Cam_27,235000,110,5,1,B + T,<NA>,<NA>,<NA>,NaN,N,1,new construction,2026,whole building


---

In [80]:
dt[dt["Street"] == "Trince delle Frasche"]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,Floor1
1,Isola Sacra (FCO) - Via Trincea delle Frasche,Trince delle Frasche,Iso_Fra_2,197000,60,2,1,T,0,1,2,O,N,1,available,2025,1.0
2,Isola Sacra (FCO) - Via Trincea delle Frasche,Trince delle Frasche,Iso_Fra_3,285000,125,4,3,B,0,1,2,B,N,1,available,2026,1.0
3,Isola Sacra (FCO) - Via Trincea delle Frasche,Trince delle Frasche,Iso_Fra_4,235000,98,4,3,G,0,1,2,B,N,2,Auction,2026,1.0


---

In [83]:
dt.loc[dt["Street"] == "Trince delle Frasche", "Street"] = "Trincea delle Frasche"

---

In [86]:
dt.loc[dt["External Area"] == "T + B + G", "External Area"] = "B + T + G"
dt.loc[dt["External Area"] == "T + B", "External Area"] = "B + T"

---

In [89]:
dt.loc[dt["Property"] == "Auction", "Property"] = "auction"

---

### 2.5 Final adjustments and corrections
This final step focuses on resolving inconsistencies identified in previous checks that required deeper validation and cross-referencing.

From the **NaN check** (Section 2.2), the feature **External Surface** showed 7 records with a value of 0 and 20 NaN values. Since the absence of an external area is already flagged in the dataset using "N" in the **External Area** column, these zero values were expected to correspond to such cases.

However, a verification showed that only one record contained "N" in **External Area**, which was already associated with a NaN value in **External Surface**. This inconsistency indicated that the remaining zero values were not meaningful and likely resulted from missing information rather than actual measurements.

Given that an external surface of 0 sqm is not a realistic scenario, these values were therefore converted to NaN to maintain consistency and avoid misleading interpretations in the analysis.

Further validation was performed through cross-column checks on **Floor**, **Levels**, and related features. This led to the identification of an inconsistent record for the estate located in "Antonio Cambriglia".

The record showed suspicious similarities with the previous entry while presenting conflicting metadata. A manual cross-check against the original listing confirmed that the values had been incorrectly recorded.

As a result, the entire row was corrected by updating all relevant fields (price, rooms, bathrooms, external area, surface, floor, levels, parking space, and construction year) based on the verified listing.

These adjustments represent targeted corrections derived from logical inconsistencies and external validation, ensuring that the dataset is both internally coherent and aligned with the source data.

In [93]:
(dt["External Area"] == "N").sum()

1

In [95]:
dt[dt["External Area"] == "N"]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,Floor1
23,Isola Sacra (FCO) - Via Trincea delle Frasche,Passo Buole,Iso_Buo_24,239000,99,4,2,N,<NA>,1,2,B,P,2,available,<NA>,1.0


In [97]:
dt[dt["External Surface"] == 0]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,Floor1
1,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_2,197000,60,2,1,T,0,1,2,O,N,1,available,2025,1.0
2,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_3,285000,125,4,3,B,0,1,2,B,N,1,available,2026,1.0
3,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_4,235000,98,4,3,G,0,1,2,B,N,2,auction,2026,1.0
36,Isola Sacra (FCO) - Via Trincea delle Frasche,Via Condino,Iso_Con_37,220000,55,3,1,T,0,2,3,O,N,1,new construction,2026,2.0
37,Isola Sacra (FCO) - Via Trincea delle Frasche,Via Condino,Iso_Con_38,174000,42,2,1,T,0,1,3,O,N,1,new construction,2026,1.0
38,Isola Sacra (FCO) - Via Trincea delle Frasche,Via Condino,Iso_Con_39,175000,43,2,1,T,0,2,3,O,N,1,new construction,2026,2.0
42,Isola Sacra (FCO) - Via Trincea delle Frasche,Via Condino,Iso_Con_43,250000,55,3,1,T,0,3,3,O,N,1,new construction,2026,3.0


In [99]:
dt[(dt["External Area"] == "N") & (dt["External Surface"] == 0)]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,Floor1


In [101]:
dt.loc[dt["External Surface"] == 0, "External Surface"] = np.nan
(dt["External Surface"].isna()).sum()

27

In [103]:
dt.loc[23, "External Surface"] = 0

In [105]:
dt.loc[[23]]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,Floor1
23,Isola Sacra (FCO) - Via Trincea delle Frasche,Passo Buole,Iso_Buo_24,239000,99,4,2,N,0,1,2,B,P,2,available,<NA>,1.0


---

In [108]:
dt[dt["Street"] == "Sei Busi"]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,Floor1
32,Isola Sacra (FCO) - Via Trincea delle Frasche,Sei Busi,Iso_Bus_33,249000,84,3,1,B,<NA>,2,4,N,N,1,new construction,2026,2.0


In [110]:
dt[dt["Parking Space"] == "N"]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,Floor1
16,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_17,259000,136,3,1,B,<NA>,1,3,N,N,1,available,1990,1.0
17,Isola Sacra (FCO) - Via Trincea delle Frasche,Alberto Brondi,Iso_Bro_18,275000,200,8,2,B + T + G,450,<NA>,2,N,N,2,missing documentation,<NA>,whole building
18,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_19,259000,123,3,2,B,<NA>,2,2,N,N,2,available,1965,2.0
24,Isola Sacra (FCO) - Via Trincea delle Frasche,Passo Buole,Iso_Buo_25,123000,77,4,1,B + T,<NA>,2,3,N,P,2,auction,1980,2.0
27,Isola Sacra (FCO) - Via Trincea delle Frasche,Ferdinando Guerci,Iso_Gue_28,195000,71,2,1,B,<NA>,1,2,N,N,2,available,<NA>,1.0
32,Isola Sacra (FCO) - Via Trincea delle Frasche,Sei Busi,Iso_Bus_33,249000,84,3,1,B,<NA>,2,4,N,N,1,new construction,2026,2.0


---

In [113]:
dt[dt["Street"] == "Antonio Cambriglia"]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,Floor1
26,Isola Sacra (FCO) - Via Trincea delle Frasche,Antonio Cambriglia,Iso_Cam_27,235000,110,5,1,B + T,<NA>,<NA>,<NA>,NaN,N,1,new construction,2026,whole building


In [115]:
#after checking the listing again, it appeared clear that the wrong entry for the estate in Antoni Cambriglia was wrong
dt.loc[26, "Price"] = 365000
dt.loc[26,"N. of Rooms"] = 4
dt.loc[26,"N. of Bathrooms"] = 2
dt.loc[26,"External Area"] = "G + T"
dt.loc[26,"External Surface"] = 100
dt.loc[26,"Floor"] = np.nan
dt.loc[26,"Levels"] = 2
dt.loc[26,"Parking Space"] = "O + B"
dt.loc[26,"Furnished"] = "N"
dt.loc[26,"Conditions"] = 1
dt.loc[26,"Property"] = "new construction"
dt.loc[26,"Construction Year"] = 2026
dt[dt["Street"] == "Antonio Cambriglia"]

,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year,Floor1
26,Isola Sacra (FCO) - Via Trincea delle Frasche,Antonio Cambriglia,Iso_Cam_27,365000,110,4,2,G + T,100,<NA>,2,O + B,N,1,new construction,2026,whole building


---

## 3. Adding new listings
At a later stage of the project, additional listings became available and were integrated into the dataset after the main cleaning process had already been completed.

These new records were generated using the previously defined class structure and then merged with the existing cleaned dataset. A quick validation check was performed on the newly added data to ensure consistency with the established format and structure.

Since the newly created dataset initially stored all columns as object type (due to the class initialization), numerical features were manually converted to the appropriate integer format (`Int64`) to maintain consistency with the rest of the dataset.

Given the limited number of new entries, minor adjustments and validations were performed manually, ensuring correctness without re-running the full cleaning pipeline.

In [122]:
new_listings = RealListing("Isola Sacra (FCO) - Via Trincea delle Frasche")

In [156]:
new_listings.characteristics(street = "Passo Buole", price = 210, mq = 60, rooms = 3, bathrooms = 1, 
                        exteriors = "T + G", ex_surface = 300,  floor = 3, levels = 3, parking = "N",
                        furnished = "N", conditions = 2, construction_year = 1995, proprieta = "available")

Careful, this street name has already been added


Do you still want to insert the record?: Y or N y


,Area,Street,ID_Number,Price,Surface,N. of Rooms,N. of Bathrooms,External Area,External Surface,Floor,Levels,Parking Space,Furnished,Conditions,Property,Construction Year
0,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea delle Frasche,Iso_Fra_1,259000,140,3,1,B,NaN,2,2,O + B,N,2,available,NaN
1,Isola Sacra (FCO) - Via Trincea delle Frasche,Edmondo Buccarelli,Iso_Buc_2,310000,140,4,2,T + G,150,1,2,O,N,2,available,NaN
2,Isola Sacra (FCO) - Via Trincea delle Frasche,Passo Buole,Iso_Buo_3,245000,125,4,1,B,NaN,2,2,O,N,2,available,NaN
3,Isola Sacra (FCO) - Via Trincea delle Frasche,Trincea dell Frasche,Iso_Fra_4,167000,56,3,1,N,0,1,2,O,N,2,available,2004.0
4,Isola Sacra (FCO) - Via Trincea delle Frasche,Giuseppe Baisi,Iso_Bai_5,340000,140,4,1,T + G,300,1,1,O + B,N,2,available,1995.0
5,Isola Sacra (FCO) - Via Trincea delle Frasche,Passo Buole,Iso_Buo_6,210000,60,3,1,T + G,300,3,3,N,N,2,available,1995.0


In [170]:
new_dt = new_listings.dataset()

In [172]:
numerical_check(new_dt, numerical_list)

,Column,Min Value,Max Value,Data Type
0,Price,167000.0,340000.0,object
1,Surface,56.0,140.0,object
2,N. of Rooms,3.0,4.0,object
3,N. of Bathrooms,1.0,2.0,object
4,External Surface,0.0,300.0,object
5,Floor,1.0,3.0,object
6,Levels,1.0,3.0,object
7,Conditions,2.0,2.0,object
8,Construction Year,1995.0,2004.0,float64


In [174]:
type(new_dt["Price"].iloc[0])

int

In [214]:
for col3 in numerical_list:
    new_dt[col3] = new_dt[col3].astype("Int64")
    print("-" * 80)
    print(f"\033[92m{col3} has been converted from Float to:\033[0m {new_dt[col3].dtype}")

--------------------------------------------------------------------------------
Price has been converted from Float to: Int64
--------------------------------------------------------------------------------
Surface has been converted from Float to: Int64
--------------------------------------------------------------------------------
N. of Rooms has been converted from Float to: Int64
--------------------------------------------------------------------------------
N. of Bathrooms has been converted from Float to: Int64
--------------------------------------------------------------------------------
External Surface has been converted from Float to: Int64
--------------------------------------------------------------------------------
Floor has been converted from Float to: Int64
--------------------------------------------------------------------------------
Levels has been converted from Float to: Int64
--------------------------------------------------------------------------------


In [192]:
numerical_check(dt, numerical_list)

,Column,Min Value,Max Value,Data Type
0,Price,123000,594000,Int64
1,Surface,42,400,Int64
2,N. of Rooms,2,10,Int64
3,N. of Bathrooms,1,4,Int64
4,External Surface,0,450,Int64
5,Floor,0,3,Int64
6,Levels,1,4,Int64
7,Conditions,1,3,Int64
8,Construction Year,1965,2026,Int64


---

In [166]:
categorical_check(new_dt, categorical_list)

,Column,Leading Spaces,Trailing Spaces,Multiple Internal Spaces,First Capital Letter
0,Area,0,0,0,5
1,Street,0,0,0,5
2,ID_Number,0,0,0,5
3,External Area,0,0,0,5
4,Parking Space,0,0,0,5
5,Furnished,0,0,0,5
6,Property,0,0,0,0


---

In [ ]:
dt = pd.concat([dt, new_dt], ignore_index=True)

In [195]:
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Area               49 non-null     object
 1   Street             49 non-null     object
 2   ID_Number          49 non-null     object
 3   Price              49 non-null     Int64 
 4   Surface            49 non-null     Int64 
 5   N. of Rooms        49 non-null     Int64 
 6   N. of Bathrooms    49 non-null     Int64 
 7   External Area      49 non-null     object
 8   External Surface   22 non-null     Int64 
 9   Floor              38 non-null     Int64 
 10  Levels             49 non-null     Int64 
 11  Parking Space      48 non-null     object
 12  Furnished          49 non-null     object
 13  Conditions         49 non-null     Int64 
 14  Property           49 non-null     object
 15  Construction Year  42 non-null     Int64 
 16  Floor1             43 non-null     object
dtyp

---

In [197]:
dt.duplicated().sum()

0

## 4. Column normalization and reorganization
Column names were standardized to improve clarity, consistency, and overall readability of the dataset. Ambiguous or less descriptive names were replaced with more explicit alternatives (e.g. inclusion of measurement units and clearer feature descriptions), ensuring that each variable can be immediately interpreted without additional context.

In parallel, columns were reorganized into a more logical structure, grouping identifiers, core property features, external characteristics, and listing-related attributes. This restructuring improves both readability and usability for subsequent analysis.

Categorical flags (e.g. **B**, **T**, **G** for external features) were intentionally preserved rather than expanded into full descriptive labels. While replacing them with full names (e.g. *Balcony*, *Terrace*, *Garden*) was considered, maintaining the compact flag format was deemed more efficient from an analytical perspective, particularly for filtering, grouping, and modeling tasks.

To ensure interpretability, a complete legend mapping these flags to their full meanings is provided in the project README.

In [201]:
new_columns = {
    "Area": "Area",
    "Street": "Street",
    "ID_Number": "ID_Listing",
    "Price": "Price",
    "Surface": "Surface sqm",
    "N. of Rooms": "Rooms",
    "N. of Bathrooms": "Bathrooms",
    "External Area": "External Features",
    "External Surface": "External Surface sqm",
    "Floor": "Floor",
    "Levels": "Levels",
    "Parking Space": "Parking",
    "Furnished": "Furnished",
    "Conditions": "Condition",
    "Property": "Listing Status",
    "Construction Year": "Construction Year"
}

In [203]:
dt.rename(columns = new_columns, inplace = True)

In [205]:
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Area                  49 non-null     object
 1   Street                49 non-null     object
 2   ID_Listing            49 non-null     object
 3   Price                 49 non-null     Int64 
 4   Surface sqm           49 non-null     Int64 
 5   Rooms                 49 non-null     Int64 
 6   Bathrooms             49 non-null     Int64 
 7   External Features     49 non-null     object
 8   External Surface sqm  22 non-null     Int64 
 9   Floor                 38 non-null     Int64 
 10  Levels                49 non-null     Int64 
 11  Parking               48 non-null     object
 12  Furnished             49 non-null     object
 13  Condition             49 non-null     Int64 
 14  Listing Status        49 non-null     object
 15  Construction Year     42 non-null     Int6

---

## 5. Export to CSV
The cleaned dataset was exported to a CSV file for further analysis and visualization.

In [216]:
dt.to_csv("Real_Listing_FCO_cleaned.csv", index = False)

***The dataset is clean, consistent and ready for use***

---
---

## 6. OMI dataset Fiumicino
This dataset contains only 4 records for a single OMI area in Fiumicino. Due to its small size, the cleaning process was performed manually through direct inspection and cross-checking with the official OMI source.

In [269]:
omi_dt = pd.read_csv("../Datasets/OMI_FCO.csv")

In [271]:
omi_dt

,Tipologia,Stato conservativo,Valori Compravendita (€/mq) - Min,Valori Compravendita (€/mq) - Max,Superficie (L/N),Valori Locazione (€/mq x mese) - Min,Valori Locazione (€/mq x mese) - Max,Superficie (L/N) - Superficie (L/N).1,Area,Province,Municipality,Fascia/Zona,Codice Zona,Microzona Catastale,Tipologia prevalente,Destinazione
0,Abitazioni civili,NORMALE,1850,2750,L,8,12,L,Isola Sacra,Rome,Fiumicino,Periferica/ISOLA SACRA-DARSENA (VIA DEL FARO),D1,0,Abitazioni Civili,Residenziale
1,Abitazioni di tipo economico,NORMALE,1750,2600,L,7,10,L,Isola Sacra,Rome,Fiumicino,Periferica/ISOLA SACRA-DARSENA (VIA DEL FARO),D1,0,Abitazioni Civili,Residenziale
2,Box,NORMALE,1100,1650,L,4,6,L,Isola Sacra,Rome,Fiumicino,Periferica/ISOLA SACRA-DARSENA (VIA DEL FARO),D1,0,Abitazioni Civili,Residenziale
3,Ville e Villini,NORMALE,2100,3000,L,8,115,L,Isola Sacra,Rome,Fiumicino,Periferica/ISOLA SACRA-DARSENA (VIA DEL FARO),D1,0,Abitazioni Civili,Residenziale


In [273]:
omi_dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 16 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   Tipologia                              4 non-null      object
 1   Stato conservativo                     4 non-null      object
 2   Valori Compravendita (€/mq) - Min      4 non-null      int64 
 3   Valori Compravendita (€/mq) - Max      4 non-null      int64 
 4   Superficie (L/N)                       4 non-null      object
 5   Valori Locazione (€/mq x mese) - Min   4 non-null      int64 
 6   Valori Locazione (€/mq x mese) - Max   4 non-null      int64 
 7   Superficie (L/N) - Superficie (L/N).1  4 non-null      object
 8   Area                                   4 non-null      object
 9   Province                               4 non-null      object
 10  Municipality                           4 non-null      object
 11  Fascia/Zona            

---

### 6.1 Correcting wrong values
A discrepancy was identified in the rent price values, where one entry appeared inconsistent compared to the others. After verifying the data against the OMI web page, the value was corrected accordingly.

In [279]:
omi_dt.iloc[3, 6] = 11.5

/var/folders/gs/d24257s15q1bzbn_vyvcxhb00000gp/T/ipykernel_39889/3719070476.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '11.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  omi_dt.iloc[3, 6] = 11.5


In [281]:
omi_dt.iloc[3, 6]

11.5

---

### 6.2 Deleting non useful features
Some columns were removed as they were either redundant or not relevant for the analysis. These include duplicated surface indicators, *Stato conservativo*, and *Destinazione*, which did not provide additional analytical value.

In [289]:
omi_dt.drop(columns = ["Stato conservativo",
                    "Superficie (L/N)", 
                    "Superficie (L/N) - Superficie (L/N).1",
                    "Destinazione", 
                   ], inplace = True)

---

### 6.3 Name normalization and column reordering
Column names were standardized to match the structure of the Rome OMI dataset, ensuring consistency across datasets. The columns were also reordered according to the predefined schema to facilitate comparison and integration.

In [293]:
fco_columns = {
    "Area": "Area",
    "Tipologia": "Property Type",
    "Valori Compravendita (€/mq) - Min": "Sale Price €/sqm Min",
    "Valori Compravendita (€/mq) - Max": "Sale Price €/sqm Max",
    "Valori Locazione (€/mq x mese) - Min": "Rent Price €/sqm per Month Min",
    "Valori Locazione (€/mq x mese) - Max": "Rent Price €/sqm per Month Max",
    "Province": "Province",
    "Municipality": "Municipality",
    "Fascia/Zona": "OMI Zone",
    "Codice Zona": "OMI Zone Code",
    "Microzona Catastale": "Cadastral Microzone",
    "Tipologia prevalente": "Dominant Property Type"
}

omi_dt = omi_dt[list(fco_columns.keys())].rename(columns=fco_columns)

In [297]:
omi_dt

,Area,Property Type,Sale Price €/sqm Min,Sale Price €/sqm Max,Rent Price €/sqm per Month Min,Rent Price €/sqm per Month Max,Province,Municipality,OMI Zone,OMI Zone Code,Cadastral Microzone,Dominant Property Type
0,Isola Sacra,Abitazioni civili,1850,2750,8,12.0,Rome,Fiumicino,Periferica/ISOLA SACRA-DARSENA (VIA DEL FARO),D1,0,Abitazioni Civili
1,Isola Sacra,Abitazioni di tipo economico,1750,2600,7,10.0,Rome,Fiumicino,Periferica/ISOLA SACRA-DARSENA (VIA DEL FARO),D1,0,Abitazioni Civili
2,Isola Sacra,Box,1100,1650,4,6.0,Rome,Fiumicino,Periferica/ISOLA SACRA-DARSENA (VIA DEL FARO),D1,0,Abitazioni Civili
3,Isola Sacra,Ville e Villini,2100,3000,8,11.5,Rome,Fiumicino,Periferica/ISOLA SACRA-DARSENA (VIA DEL FARO),D1,0,Abitazioni Civili


---

### 6.4 Exporting the Dataset to CSV file

In [301]:
omi_dt.to_csv("OMI_FCO_Cleaned.csv", index = False)